# CoffeeRen — Fine-tuning YOLO26n (v6 -> v7, incorporando fotos de la camara ELP)

Continua el entrenamiento desde los pesos de v6 (no entrena desde cero) agregando el dataset combinado (3600 fotos del celular + nuevas fotos de la ELP) y augmentation HSV.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

In [ ]:
!pip install ultralytics roboflow -q

## 3. Montar Drive (necesario para cargar el best.pt de v6)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Descargar dataset v7 desde Roboflow

La API key NO va en texto plano: en Colab, icono de llave (barra lateral izquierda) -> "Add new secret" -> nombre `ROBOFLOW_API_KEY`, valor tu key -> activar acceso a este notebook.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("andersonbacach").project("coffeeren_od")
version = project.version(7)  # v7 = v6 (celular) + fotos nuevas de la ELP, ya etiquetadas en Roboflow
dataset = version.download("yolov8")
print(f"Dataset descargado en: {dataset.location}")

## 5. Verificar estructura del dataset

In [ ]:
import os, yaml

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml, "r") as f:
    data = yaml.safe_load(f)
print(f"Clases ({data['nc']}): {data['names']}")

## 6. Fine-tuning: partir del best.pt de v6, no de COCO

In [ ]:
from ultralytics import YOLO

modelo_base_v6 = "/content/drive/MyDrive/CoffeeRen/modelos/yolo26n_resultados_v6_2cls/best.pt"
model = YOLO(modelo_base_v6)

results = model.train(
    data=data_yaml,
    epochs=50,          # antes 100 - ya no parte de cero, converge mas rapido
    imgsz=640,           # NO cambiar - coincide con la inferencia en produccion
    batch=32,
    patience=15,         # antes 20 - early stopping mas agil en fine-tuning
    workers=2,
    device=0,
    project="coffeeren",
    name="yolo26n_v7_2cls_ft_elp",
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,           # bajo a proposito: estas afinando, no entrenando desde cero
    seed=42,
    verbose=True,
    hsv_h=0.02,          # jitter de tono - esto es lo que faltaba en el augmentation original
    hsv_s=0.7,           # jitter de saturacion
    hsv_v=0.4,           # jitter de brillo
)
print("Entrenamiento completado")

## 7. Evaluar, ver curvas y guardar

In [ ]:
from IPython.display import Image, display
import shutil, glob

base = "/content/runs/detect/coffeeren/yolo26n_v7_2cls_ft_elp"

best_model = YOLO(f"{base}/weights/best.pt")
metrics = best_model.val(data=data_yaml, split="test", device=0)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP75:     {metrics.box.map75:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

display(Image(f"{base}/results.png"))
display(Image(f"{base}/confusion_matrix_normalized.png"))

test_images = glob.glob(os.path.join(dataset.location, "test/images/*.jpg"))
result = best_model.predict(
    source=test_images[0],
    conf=0.5,
    save=True,
    project="coffeeren",
    name="prediccion_test_26n_v7_2cls"
)
pred_img = glob.glob("/content/runs/detect/coffeeren/prediccion_test_26n_v7_2cls/*.jpg")[0]
display(Image(pred_img))

# Guardar en Drive (carpeta v7, separada de v6 - no se sobreescribe el modelo anterior)
save_path = "/content/drive/MyDrive/CoffeeRen/modelos/yolo26n_resultados_v7_2cls_ft_elp"
os.makedirs(save_path, exist_ok=True)
shutil.copy(f"{base}/weights/best.pt", save_path)
shutil.copy(f"{base}/weights/last.pt", save_path)
shutil.copy(f"{base}/results.png", save_path)
shutil.copy(f"{base}/confusion_matrix_normalized.png", save_path)
print(f"Guardado en: {save_path}")

## 8. Validacion especifica contra fotos SOLO de la ELP

Carpeta aparte en Drive con fotos de la ELP que NO se metieron en Roboflow (guardadas aparte justo para esta comprobacion) - asi se ve si el problema real mejoro, no solo el mAP del set combinado.

In [ ]:
elp_val_dir = "/content/drive/MyDrive/CoffeeRen/validacion_elp/*.jpg"
elp_images = glob.glob(elp_val_dir)
if elp_images:
    result_elp = best_model.predict(
        source=elp_images,
        conf=0.35,
        save=True,
        project="coffeeren",
        name="validacion_solo_elp"
    )
    print(f"{len(elp_images)} imagenes ELP evaluadas - revisa coffeeren/validacion_solo_elp")
else:
    print("No se encontraron imagenes en validacion_elp/ - sube algunas para esta comprobacion")